In [31]:
import pandas as pd
import numpy as np

# crosstab report with neighborhoof and year
yearly = pd.crosstab(index=df['Neighborhood'], columns=df['Arrest_Year'])

# check years are present
for year in yearly:
    if year not in yearly.columns:
        yearly[year] = 0

# calculate percentages
with np.errstate(divide='ignore', invalid='ignore'):
    growth_24_25 = ((yearly[2025] - yearly[2024]) / yearly[2024]) * 100
    growth_25_26 = ((yearly[2026] - yearly[2025]) / yearly[2025]) * 100

    # check dividing by 0
    growth_24_25 = growth_24_25.replace([np.inf, -np.inf], np.nan).fillna(0)
    growth_25_26 = growth_25_26.replace([np.inf, -np.inf], np.nan).fillna(0)

# 2024 to 2025 trend percentage
yearly['2024-2025 Trend'] = growth_24_25

# trend percentage
yearly['Total Trend'] = (growth_24_25 + growth_25_26) / 2

# "total" column
yearly['Total'] = yearly[2024] + yearly[2025] + yearly[2026]

# format trend column
def format_trend(val):
    if pd.isna(val) or val == 0:
        return "0.0%"
    return f"+{val:.1f}%" if val > 0 else f"{val:.1f}%"

# sort rows by crime
report = yearly.sort_values(by='Total', ascending=False)
report['2024-2025 Trend'] = report['2024-2025 Trend'].apply(format_trend)
report['Total Trend'] = report['Total Trend'].apply(format_trend)

# diable max rows
pd.set_option('display.max_rows', None)
report

Arrest_Year,2024,2025,2026,2024-2025 Trend,Total Trend,Total
Neighborhood,,,,,,
Bluff,2631,2431,3049,-7.6%,+8.9%,8111
Central Business District,2346,2749,2780,+17.2%,+9.2%,7875
South Side Flats,1299,1483,1389,+14.2%,+3.9%,4171
East Allegheny,692,1054,742,+52.3%,+11.4%,2488
Carrick,496,603,446,+21.6%,-2.2%,1545
East Liberty,454,482,599,+6.2%,+15.2%,1535
Knoxville,392,650,365,+65.8%,+11.0%,1407
Homewood South,585,440,348,-24.8%,-22.8%,1373
Lincoln-Lemington-Belmar,391,503,247,+28.6%,-11.1%,1141


In [42]:
# calculate divisor
if report['2024-2025 Trend'].dtype == 'O':
    pct_change = report['2024-2025 Trend'].astype(str).str.replace('%', '').str.replace('+', '').astype(float)
else:
    pct_change = report['2024-2025 Trend']

report['Divisor Baseline'] = 100 - pct_change

# create safety score
report['Safety Score'] = report['Divisor Baseline'] / report['Total']

# sort by score
report = report.sort_values(by='Safety Score', ascending=False)

# display
report[['Rank', 'Total', '2024-2025 Trend', 'Safety Score']]

Arrest_Year,Rank,Total,2024-2025 Trend,Safety Score
Neighborhood,,,,
St. Clair,1,17,-70.0%,10.000000
Ridgemont,2,18,-12.5%,6.250000
New Homestead,3,24,0.0%,4.166667
Windgap,4,70,-58.3%,2.261429
Oakwood,5,51,-12.5%,2.205882
Swisshelm Park,6,25,+50.0%,2.000000
Esplen,7,92,-45.5%,1.581522
Friendship,8,50,+21.1%,1.578000
Westwood,9,109,-42.6%,1.308257
